# LeetCode 1331 — Rank Transform of an Array bằng Python

**Link đề:** https://leetcode.com/problems/rank-transform-of-an-array/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Sorting (nén toạ độ — *coordinate compression*)

```python
class Solution:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **hạng (rank)** theo kiểu "dense rank": phần tử bằng nhau cùng hạng, hạng liên tiếp không nhảy cóc.
2. Ôn và dùng thành thạo: `set`, `sorted`, `dict`, `enumerate(start=1)`, dict comprehension, `bisect`.
3. Đi từ brute force `O(n²)` → sắp xếp + `dict` `O(n log n)` → các biến thể (`bisect`, sắp xếp chỉ số).
4. Trace, chứng minh, Time/Space, test đối chiếu ngẫu nhiên.
5. Nhận ra pattern **nén toạ độ** — dùng nhiều trong bài khó hơn (Fenwick tree, đếm nghịch thế...).

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho mảng số nguyên `arr`, thay mỗi phần tử bằng **hạng** của nó. Luật hạng:

- Hạng là số nguyên bắt đầu từ `1`.
- Phần tử lớn hơn có hạng lớn hơn. Hai phần tử **bằng nhau** có **cùng hạng**.
- Hạng phải **nhỏ nhất có thể**.

| `arr` | Output |
|---|---|
| `[40,10,20,30]` | `[4,1,2,3]` |
| `[100,100,100]` | `[1,1,1]` |
| `[37,12,28,9,100,56,80,5,12]` | `[5,3,4,2,8,6,7,1,3]` |

**Constraints:** `0 <= arr.length <= 10^5`, `-10^9 <= arr[i] <= 10^9`.

Chú ý: mảng có thể **rỗng** → trả về `[]`.

Câu hỏi trước khi code:
- "Hạng nhỏ nhất có thể" nghĩa là gì? Với `[10, 20, 20, 30]`, hạng của `30` là `3` hay `4`?
- Với `n = 10^5`, cách `O(n²)` có chạy kịp không?

## 2. Hiểu "hạng" bằng tay

Hint của đề: **hạng = (số giá trị KHÁC NHAU nhỏ hơn nó) + 1**.

Ví dụ `[10, 20, 20, 30]`:
- Các giá trị khác nhau, sắp xếp: `10, 20, 30`.
- `10` → có 0 giá trị nhỏ hơn → hạng 1.
- `20` → có 1 giá trị (`10`) nhỏ hơn → hạng 2.
- `30` → có 2 giá trị (`10, 20`) nhỏ hơn → hạng **3** (không phải 4, vì hai số `20` chỉ tính một lần).

Kiểu hạng này gọi là **dense rank** (hạng dày, không có lỗ hổng).

In [1]:
arr = [10, 20, 20, 30]
distinct_sorted = sorted(set(arr))
print("giá trị khác nhau đã sắp xếp:", distinct_sorted)
for x in arr:
    smaller = [v for v in distinct_sorted if v < x]
    print(f"x={x}: các giá trị khác nhau nhỏ hơn = {smaller} -> hạng {len(smaller) + 1}")

giá trị khác nhau đã sắp xếp: [10, 20, 30]
x=10: các giá trị khác nhau nhỏ hơn = [] -> hạng 1
x=20: các giá trị khác nhau nhỏ hơn = [10] -> hạng 2
x=20: các giá trị khác nhau nhỏ hơn = [10] -> hạng 2
x=30: các giá trị khác nhau nhỏ hơn = [10, 20] -> hạng 3


## 3. Nền tảng Python

### 3.1. `set` — bỏ trùng

`set([100, 100, 100])` → `{100}`. Thứ tự trong `set` **không được đảm bảo**, nên muốn có thứ tự phải sắp xếp.

In [2]:
print(set([100, 100, 100]))
print(set([37, 12, 28, 9, 12]))
print(len(set([37, 12, 28, 9, 12])), "giá trị khác nhau")

{100}
{9, 28, 12, 37}
4 giá trị khác nhau


### 3.2. `sorted(iterable)` — trả về **list mới** đã sắp tăng dần

`sorted` nhận list, set, ... và trả list mới, **không** sửa dữ liệu gốc (khác với `list.sort()` sửa tại chỗ và trả `None`).

In [3]:
arr = [40, 10, 20, 30]
print("sorted(arr):", sorted(arr), "| arr vẫn là:", arr)
print("sorted(set):", sorted({3, 1, 2}))
tmp = arr.copy()
print("tmp.sort() trả về:", tmp.sort(), "| tmp sau sort:", tmp)

sorted(arr): [10, 20, 30, 40] | arr vẫn là: [40, 10, 20, 30]
sorted(set): [1, 2, 3]
tmp.sort() trả về: None | tmp sau sort: [10, 20, 30, 40]


### 3.3. `dict` — bảng tra giá trị → hạng

`rank = {10: 1, 20: 2}` ; `rank[20]` → `2`. Tra cứu trung bình `O(1)`.

### 3.4. `enumerate(iterable, start=1)` — vừa lấy phần tử vừa lấy số thứ tự bắt đầu từ 1

In [4]:
for i, v in enumerate([10, 20, 30], start=1):
    print("thứ tự", i, "giá trị", v)

rank = {}
for i, v in enumerate([10, 20, 30], start=1):
    rank[v] = i
print("dict:", rank, "| rank[20] =", rank[20])

thứ tự 1 giá trị 10
thứ tự 2 giá trị 20
thứ tự 3 giá trị 30
dict: {10: 1, 20: 2, 30: 3} | rank[20] = 2


### 3.5. Dict comprehension và list comprehension

- `{v: i for i, v in enumerate(lst, 1)}` tạo dict trong 1 dòng.
- `[rank[x] for x in arr]` tạo list bằng cách tra từng phần tử.

In [5]:
lst = [10, 20, 30]
rank = {v: i for i, v in enumerate(lst, 1)}
print(rank)
print([rank[x] for x in [30, 10, 20, 20]])

{10: 1, 20: 2, 30: 3}
[3, 1, 2, 2]


## 4. Cách trực giác / baseline — đếm trực tiếp `O(n²)`

Làm đúng theo định nghĩa: với **mỗi** `x`, đếm số giá trị khác nhau nhỏ hơn `x`.

In [6]:
def rank_brute(arr: list[int]) -> list[int]:
    distinct = set(arr)
    result = []
    for x in arr:
        count_smaller = 0
        for v in distinct:
            if v < x:
                count_smaller += 1
        result.append(count_smaller + 1)
    return result

print(rank_brute([40, 10, 20, 30]))
print(rank_brute([100, 100, 100]))
print(rank_brute([37, 12, 28, 9, 100, 56, 80, 5, 12]))
print(rank_brute([]))

[4, 1, 2, 3]
[1, 1, 1]
[5, 3, 4, 2, 8, 6, 7, 1, 3]
[]


### 4.1. Nút thắt

Với mỗi trong `n` phần tử, ta duyệt lại tới `n` giá trị khác nhau → `O(n²)`. Với `n = 10^5`, đó là khoảng `10^10` phép so sánh — **quá chậm** (Time Limit Exceeded).

Công việc bị lặp: ta **đếm đi đếm lại** "bao nhiêu giá trị nhỏ hơn" cho từng `x`. Nếu sắp xếp các giá trị khác nhau **một lần**, thì vị trí của `x` trong danh sách đã sắp xếp **chính là** số giá trị nhỏ hơn nó!

In [7]:
import time
big = list(range(3000, 0, -1))
t0 = time.perf_counter(); rank_brute(big); t1 = time.perf_counter()
print(f"brute với n=3000 mất khoảng {t1 - t0:.2f}s -> n=10^5 sẽ lâu hơn ~1000 lần")

brute với n=3000 mất khoảng 0.24s -> n=10^5 sẽ lâu hơn ~1000 lần


## 5. Insight — sắp xếp các giá trị khác nhau, vị trí chính là hạng

`distinct_sorted = sorted(set(arr))`. Phần tử ở vị trí `i` (0-based) có đúng `i` giá trị khác nhau nhỏ hơn nó ⇒ hạng `i + 1`.

Lưu vào `dict` **giá trị → hạng**, rồi tra cho từng phần tử của `arr`. Đây là kỹ thuật **nén toạ độ** (coordinate compression): ánh xạ các giá trị lớn tuỳ ý (`-10^9..10^9`) về `1..k` mà giữ nguyên thứ tự.

In [8]:
arr = [37, 12, 28, 9, 100, 56, 80, 5, 12]
distinct_sorted = sorted(set(arr))
print("distinct_sorted:", distinct_sorted)
for i, v in enumerate(distinct_sorted):
    print(f"  vị trí {i}: {v} có {i} giá trị nhỏ hơn -> hạng {i + 1}")

distinct_sorted: [5, 9, 12, 28, 37, 56, 80, 100]
  vị trí 0: 5 có 0 giá trị nhỏ hơn -> hạng 1
  vị trí 1: 9 có 1 giá trị nhỏ hơn -> hạng 2
  vị trí 2: 12 có 2 giá trị nhỏ hơn -> hạng 3
  vị trí 3: 28 có 3 giá trị nhỏ hơn -> hạng 4
  vị trí 4: 37 có 4 giá trị nhỏ hơn -> hạng 5
  vị trí 5: 56 có 5 giá trị nhỏ hơn -> hạng 6
  vị trí 6: 80 có 6 giá trị nhỏ hơn -> hạng 7
  vị trí 7: 100 có 7 giá trị nhỏ hơn -> hạng 8


## 6. Lời giải chính — `sorted(set)` + `dict`, giảng từng đoạn

**Đoạn 1:** `distinct_sorted = sorted(set(arr))` — bỏ trùng rồi sắp xếp.

**Đoạn 2:** `rank = {v: i for i, v in enumerate(distinct_sorted, 1)}` — giá trị → hạng.

**Đoạn 3:** `return [rank[x] for x in arr]` — giữ đúng **thứ tự gốc** của `arr`.

Mảng rỗng: `set([])` rỗng, `sorted` ra `[]`, dict rỗng, list kết quả rỗng — tự động đúng, không cần `if`.

In [9]:
def rank_sort_dict(arr: list[int]) -> list[int]:
    distinct_sorted = sorted(set(arr))
    rank = {v: i for i, v in enumerate(distinct_sorted, 1)}
    return [rank[x] for x in arr]

print(rank_sort_dict([40, 10, 20, 30]))
print(rank_sort_dict([100, 100, 100]))
print(rank_sort_dict([37, 12, 28, 9, 100, 56, 80, 5, 12]))
print(rank_sort_dict([]))

[4, 1, 2, 3]
[1, 1, 1]
[5, 3, 4, 2, 8, 6, 7, 1, 3]
[]


## 7. TRACE từng bước

Trace `arr = [40, 10, 20, 30, 20]`:

**Giai đoạn 1 — dựng bảng:**

| bước | v (từ distinct_sorted) | hạng | rank sau bước |
|---|---|---|---|
| 1 | 10 | 1 | `{10:1}` |
| 2 | 20 | 2 | `{10:1, 20:2}` |
| 3 | 30 | 3 | `{..., 30:3}` |
| 4 | 40 | 4 | `{..., 40:4}` |

**Giai đoạn 2 — tra theo thứ tự gốc:** `40→4, 10→1, 20→2, 30→3, 20→2` ⇒ `[4,1,2,3,2]`.

In [10]:
def trace_rank(arr: list[int]) -> list[int]:
    print("arr =", arr)
    distinct_sorted = sorted(set(arr))
    print("set(arr) rồi sorted ->", distinct_sorted)
    rank = {}
    for i, v in enumerate(distinct_sorted, 1):
        rank[v] = i
        print(f"  gán rank[{v}] = {i} -> rank = {rank}")
    result = []
    for x in arr:
        result.append(rank[x])
        print(f"  tra x={x}: rank[x]={rank[x]} -> result = {result}")
    return result

trace_rank([40, 10, 20, 30, 20])

arr = [40, 10, 20, 30, 20]
set(arr) rồi sorted -> [10, 20, 30, 40]
  gán rank[10] = 1 -> rank = {10: 1}
  gán rank[20] = 2 -> rank = {10: 1, 20: 2}
  gán rank[30] = 3 -> rank = {10: 1, 20: 2, 30: 3}
  gán rank[40] = 4 -> rank = {10: 1, 20: 2, 30: 3, 40: 4}
  tra x=40: rank[x]=4 -> result = [4]
  tra x=10: rank[x]=1 -> result = [4, 1]
  tra x=20: rank[x]=2 -> result = [4, 1, 2]
  tra x=30: rank[x]=3 -> result = [4, 1, 2, 3]
  tra x=20: rank[x]=2 -> result = [4, 1, 2, 3, 2]


[4, 1, 2, 3, 2]

## 8. Code nộp LeetCode

In [11]:
class Solution:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        rank = {v: i for i, v in enumerate(sorted(set(arr)), 1)}
        return [rank[x] for x in arr]

sol = Solution()
print(sol.arrayRankTransform([40, 10, 20, 30]))
print(sol.arrayRankTransform([100, 100, 100]))
print(sol.arrayRankTransform([37, 12, 28, 9, 100, 56, 80, 5, 12]))

[4, 1, 2, 3]
[1, 1, 1]
[5, 3, 4, 2, 8, 6, 7, 1, 3]


## 9. Các lời giải khác

### Cách 3 — sắp xếp toàn bộ (có trùng), gán hạng khi giá trị đổi

Không dùng `set`: sắp xếp `arr` (bản sao), đi từ trái sang, **chỉ tăng hạng khi gặp giá trị mới** (khác giá trị trước).

In [12]:
def rank_sort_scan(arr: list[int]) -> list[int]:
    rank = {}
    current = 0
    for v in sorted(arr):
        if v not in rank:        # giá trị mới -> hạng tiếp theo
            current += 1
            rank[v] = current
    return [rank[x] for x in arr]

print(rank_sort_scan([40, 10, 20, 30, 20]))
print(rank_sort_scan([100, 100, 100]))

[4, 1, 2, 3, 2]
[1, 1, 1]


### Cách 4 — tìm kiếm nhị phân `bisect`

Module chuẩn `bisect`: `bisect_left(a, x)` trả về **vị trí đầu tiên** trong list đã sắp xếp `a` mà ta có thể chèn `x` vào mà vẫn giữ thứ tự — cũng chính là **số phần tử nhỏ hơn `x`**. Tốn `O(log k)` mỗi lần.

Vậy hạng = `bisect_left(distinct_sorted, x) + 1`. Không cần `dict`.

In [13]:
from bisect import bisect_left

distinct_sorted = [10, 20, 30, 40]
for x in [10, 25, 30, 40, 5]:
    print(f"bisect_left({distinct_sorted}, {x}) = {bisect_left(distinct_sorted, x)}")

def rank_bisect(arr: list[int]) -> list[int]:
    ds = sorted(set(arr))
    return [bisect_left(ds, x) + 1 for x in arr]

print(rank_bisect([37, 12, 28, 9, 100, 56, 80, 5, 12]))

bisect_left([10, 20, 30, 40], 10) = 0
bisect_left([10, 20, 30, 40], 25) = 2
bisect_left([10, 20, 30, 40], 30) = 2
bisect_left([10, 20, 30, 40], 40) = 3
bisect_left([10, 20, 30, 40], 5) = 0
[5, 3, 4, 2, 8, 6, 7, 1, 3]


### Cách 5 — sắp xếp **chỉ số** (không cần dict)

Sắp xếp các chỉ số `0..n-1` theo giá trị `arr[i]`: `sorted(range(n), key=lambda i: arr[i])`. Đi theo thứ tự đó, gán hạng trực tiếp vào `result[i]`; tăng hạng khi giá trị đổi.

`key=` cho `sorted` biết "so sánh theo cái gì". `lambda i: arr[i]` là hàm vô danh trả `arr[i]`.

In [14]:
def rank_index_sort(arr: list[int]) -> list[int]:
    n = len(arr)
    order = sorted(range(n), key=lambda i: arr[i])
    result = [0] * n
    rank = 0
    prev = None
    for i in order:
        if arr[i] != prev:
            rank += 1
            prev = arr[i]
        result[i] = rank
    return result

arr = [40, 10, 20, 30, 20]
print("thứ tự chỉ số theo giá trị:", sorted(range(len(arr)), key=lambda i: arr[i]))
print(rank_index_sort(arr))
print(rank_index_sort([]))

thứ tự chỉ số theo giá trị: [1, 2, 4, 3, 0]
[4, 1, 2, 3, 2]
[]


Cẩn thận: `prev = None` ban đầu đảm bảo phần tử đầu tiên luôn "khác prev". Nếu viết `prev = 0` sẽ **sai** khi phần tử nhỏ nhất bằng `0` (xem mục lỗi).

### Code nộp LeetCode cho các cách khác

In [15]:
class SolutionBisect:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        ds = sorted(set(arr))
        return [bisect_left(ds, x) + 1 for x in arr]

class SolutionScan:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        rank = {}
        for v in sorted(arr):
            if v not in rank:
                rank[v] = len(rank) + 1      # len(rank) = số giá trị đã gán
        return [rank[x] for x in arr]

class SolutionIndex:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        result = [0] * len(arr)
        rank, prev = 0, None
        for i in sorted(range(len(arr)), key=arr.__getitem__):
            if arr[i] != prev:
                rank, prev = rank + 1, arr[i]
            result[i] = rank
        return result

for cls in [SolutionBisect, SolutionScan, SolutionIndex]:
    print(cls.__name__, cls().arrayRankTransform([37, 12, 28, 9, 100, 56, 80, 5, 12]))

SolutionBisect [5, 3, 4, 2, 8, 6, 7, 1, 3]
SolutionScan [5, 3, 4, 2, 8, 6, 7, 1, 3]
SolutionIndex [5, 3, 4, 2, 8, 6, 7, 1, 3]


`arr.__getitem__` là chính hàm mà `arr[i]` gọi ngầm — dùng làm `key` thay cho `lambda i: arr[i]`.

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | đếm giá trị khác nhau nhỏ hơn | `O(n·k)` ≈ `O(n²)` | `O(k)` | TLE với `n = 10^5` |
| 2. `sorted(set)` + dict | vị trí sau sắp xếp = hạng | `O(n log n)` | `O(n)` | lời giải chính, gọn |
| 3. Sắp xếp có trùng + dict | tăng hạng khi gặp giá trị mới | `O(n log n)` | `O(n)` | không cần set |
| 4. `bisect` | hạng = số phần tử nhỏ hơn + 1 | `O(n log n)` | `O(k)` | không cần dict |
| 5. Sắp xếp chỉ số | gán thẳng vào `result[i]` | `O(n log n)` | `O(n)` | tổng quát, giữ chỉ số |

`n = len(arr)`, `k` = số giá trị khác nhau (`k <= n`).

## 10. Vì sao lời giải chính đúng?

Gọi `ds = sorted(set(arr))` = `[v1 < v2 < ... < vk]` (tăng **nghiêm ngặt** vì đã bỏ trùng).

- Với `x = v_i`, các giá trị khác nhau nhỏ hơn `x` đúng là `v1..v_{i-1}`: có `i - 1` giá trị ⇒ hạng `i`. Khớp với `rank[v_i] = i`.
- Kiểm tra 3 luật: (a) hạng bắt đầu từ 1 ✔; (b) `x < y` ⇒ `x` đứng trước `y` trong `ds` ⇒ hạng nhỏ hơn; bằng nhau ⇒ cùng một key ⇒ cùng hạng ✔; (c) các hạng dùng đúng `1..k`, **không có lỗ hổng** — không thể nhỏ hơn vì `k` giá trị khác nhau cần ít nhất `k` hạng khác nhau ✔.
- Bước cuối duyệt `arr` theo thứ tự gốc nên vị trí được giữ nguyên ✔. ∎

### 10.1. Độ phức tạp

- **Time:** `set` `O(n)` + `sorted` `O(k log k)` + dựng dict `O(k)` + tra `O(n)` ⇒ `O(n log n)` (xấu nhất `k = n`).
- **Space:** `O(n)` cho set/list/dict và output.

Không thể làm tốt hơn `O(n log n)` bằng so sánh nói chung, vì từ hạng ta suy ra được thứ tự sắp xếp.

## 11. Bộ test

Mẫu đề, rỗng, một phần tử, toàn trùng, số âm, biên `±10^9`, và **đối chiếu** mọi cách với brute force trên dữ liệu ngẫu nhiên (cố tình nhiều giá trị trùng).

In [16]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS -> {actual}")

check("ví dụ 1", sol.arrayRankTransform([40, 10, 20, 30]), [4, 1, 2, 3])
check("ví dụ 2", sol.arrayRankTransform([100, 100, 100]), [1, 1, 1])
check("ví dụ 3", sol.arrayRankTransform([37, 12, 28, 9, 100, 56, 80, 5, 12]), [5, 3, 4, 2, 8, 6, 7, 1, 3])
check("rỗng", sol.arrayRankTransform([]), [])
check("một phần tử", sol.arrayRankTransform([-7]), [1])
check("số âm + 0", sol.arrayRankTransform([0, -1, 0, -5]), [3, 2, 3, 1])
check("biên ±10^9", sol.arrayRankTransform([10**9, -10**9, 0]), [3, 1, 2])

ví dụ 1          PASS -> [4, 1, 2, 3]
ví dụ 2          PASS -> [1, 1, 1]
ví dụ 3          PASS -> [5, 3, 4, 2, 8, 6, 7, 1, 3]
rỗng             PASS -> []
một phần tử      PASS -> [1]
số âm + 0        PASS -> [3, 2, 3, 1]
biên ±10^9       PASS -> [3, 1, 2]


In [17]:
import random
rng = random.Random(1331)
methods = [rank_sort_dict, rank_sort_scan, rank_bisect, rank_index_sort,
           sol.arrayRankTransform, SolutionBisect().arrayRankTransform,
           SolutionScan().arrayRankTransform, SolutionIndex().arrayRankTransform]
for _ in range(400):
    arr = [rng.randint(-5, 5) for _ in range(rng.randint(0, 25))]
    expected = rank_brute(arr)
    for f in methods:
        assert f(arr) == expected, (f, arr)
print("400 mảng ngẫu nhiên: mọi cách khớp brute force")

big = [rng.randint(-10**9, 10**9) for _ in range(10**5)]
t0 = time.perf_counter(); out = sol.arrayRankTransform(big); t1 = time.perf_counter()
print(f"n = 10^5: lời giải chính chạy {t1 - t0:.3f}s, hạng lớn nhất = {max(out)} (= số giá trị khác nhau {len(set(big))})")

400 mảng ngẫu nhiên: mọi cách khớp brute force
n = 10^5: lời giải chính chạy 0.057s, hạng lớn nhất = 99999 (= số giá trị khác nhau 99999)


## 12. Lỗi người mới hay gặp

### Lỗi 1: Không bỏ trùng → hạng có "lỗ hổng"

Dùng vị trí trong `sorted(arr)` (có trùng) thì giá trị trùng chiếm nhiều vị trí, làm hạng nhảy cóc.

In [18]:
def wrong_no_dedup(arr):
    rank = {}
    for i, v in enumerate(sorted(arr), 1):
        rank[v] = i                      # giá trị trùng bị ghi đè bằng vị trí sau cùng
    return [rank[x] for x in arr]

print("sai :", wrong_no_dedup([10, 20, 20, 30]), "| đúng:", rank_sort_dict([10, 20, 20, 30]))

sai : [1, 3, 3, 4] | đúng: [1, 2, 2, 3]


### Lỗi 2: Quên sắp xếp sau khi `set`

`set` không có thứ tự đảm bảo; `enumerate(set(arr))` cho hạng ngẫu nhiên theo cách Python lưu set.

### Lỗi 3: `prev = 0` trong cách sắp xếp chỉ số

Nếu phần tử nhỏ nhất là `0`, nó bị coi là "bằng prev" → không tăng hạng → được hạng `0`.

In [19]:
def wrong_prev_zero(arr):
    result = [0] * len(arr)
    rank, prev = 0, 0                    # sai: nên là None
    for i in sorted(range(len(arr)), key=lambda i: arr[i]):
        if arr[i] != prev:
            rank, prev = rank + 1, arr[i]
        result[i] = rank
    return result

print("sai :", wrong_prev_zero([0, 5, 0]), "| đúng:", rank_index_sort([0, 5, 0]))
print("thứ tự lặp của set có thể khác thứ tự tăng:", list({3, -1, 2, 100, -50}))

sai : [0, 1, 0] | đúng: [1, 2, 1]
thứ tự lặp của set có thể khác thứ tự tăng: [2, 3, 100, -1, -50]


### Lỗi 4: Dùng `arr.sort()` làm hỏng mảng gốc

`arr.sort()` sửa **tại chỗ**. Nếu sau đó tra `[rank[x] for x in arr]` thì ta trả về hạng theo thứ tự **đã sắp xếp**, không phải thứ tự gốc.

In [20]:
def wrong_sort_inplace(arr):
    arr.sort()                           # phá thứ tự gốc!
    rank = {v: i for i, v in enumerate(dict.fromkeys(arr), 1)}
    return [rank[x] for x in arr]

print("sai :", wrong_sort_inplace([40, 10, 20, 30]), "| đúng:", rank_sort_dict([40, 10, 20, 30]))

sai : [1, 2, 3, 4] | đúng: [4, 1, 2, 3]


(`dict.fromkeys(arr)` tạo dict với key là các phần tử của `arr`, **giữ thứ tự xuất hiện**, bỏ trùng — một cách khác để khử trùng có thứ tự.)

## 13. Mẹo nhận dạng pattern

- "Thay mỗi phần tử bằng **thứ hạng / thứ tự tương đối**" ⇒ **nén toạ độ**: `sorted(set(arr))` + dict.
- Giá trị rất lớn (`10^9`) nhưng chỉ quan tâm **thứ tự** ⇒ nén về `1..k` để dùng làm chỉ số mảng (Fenwick tree, BIT, segment tree).
- Cần "số phần tử nhỏ hơn `x` trong tập đã sắp xếp" ⇒ `bisect_left`.
- Có trùng và cần hạng không lỗ hổng ⇒ nhớ **bỏ trùng** trước.

## 14. Biến thể

1. **Hạng kiểu thi đấu** (competition rank, "1224"): hạng = số phần tử (tính cả trùng) nhỏ hơn + 1 → dùng `bisect_left(sorted(arr), x) + 1`.
2. **Hạng giảm dần** (lớn nhất hạng 1): sắp xếp `reverse=True`.
3. **LeetCode 1632 — Rank Transform of a Matrix** (Hard): cùng tên nhưng có ràng buộc theo hàng/cột.
4. **LeetCode 506 — Relative Ranks**: gán huy chương theo thứ hạng.

In [21]:
def competition_rank(arr):
    s = sorted(arr)
    return [bisect_left(s, x) + 1 for x in arr]

def dense_rank_desc(arr):
    rank = {v: i for i, v in enumerate(sorted(set(arr), reverse=True), 1)}
    return [rank[x] for x in arr]

arr = [10, 20, 20, 30]
print("dense      :", rank_sort_dict(arr))
print("competition:", competition_rank(arr))
print("dense giảm :", dense_rank_desc(arr))

dense      : [1, 2, 2, 3]
competition: [1, 2, 2, 4]
dense giảm : [3, 2, 2, 1]


## 15. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Hạng dense của `[5, -2, 5, 0, 7, -2]`?

**Bài 2 (đoán output):** `sorted({3, 1, 2})`, `bisect_left([1, 3, 5], 4)`, `list(enumerate("ab", 1))`.

**Bài 3 (tự viết):** `exercise_rank(arr)` theo cách `sorted(set)` + dict.

**Bài 4 (biến thể):** `exercise_competition_rank(arr)` — hạng kiểu thi đấu `1224`.

**Bài 5 (biến thể):** `exercise_rank_desc(arr)` — dense rank, giá trị **lớn nhất** hạng 1.

**Bài 6 (mini challenge):** `exercise_count_smaller_distinct(arr)` — trả về list, phần tử thứ `i` là **số giá trị khác nhau** nhỏ hơn `arr[i]` (gợi ý: liên hệ với hạng).

In [22]:
def exercise_rank(arr: list[int]) -> list[int]:
    # TODO
    pass

def exercise_competition_rank(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_rank([40, 10, 20, 30]) == [4, 1, 2, 3]
# assert exercise_competition_rank([10, 20, 20, 30]) == [1, 2, 2, 4]

In [23]:
def exercise_rank_desc(arr: list[int]) -> list[int]:
    # TODO
    pass

def exercise_count_smaller_distinct(arr: list[int]) -> list[int]:
    # TODO
    pass

# assert exercise_rank_desc([10, 20, 20, 30]) == [3, 2, 2, 1]
# assert exercise_count_smaller_distinct([10, 20, 20, 30]) == [0, 1, 1, 2]

## 16. Gợi ý 3 mức

**Bài 1** — (1) Bỏ trùng: `{-2, 0, 5, 7}`. (2) Sắp xếp và đánh số từ 1. (3) Tra lại theo thứ tự gốc.

**Bài 3** — (1) Ba dòng. (2) `sorted(set(arr))`. (3) Dict comprehension + list comprehension.

**Bài 4** — (1) Không bỏ trùng. (2) "Số phần tử nhỏ hơn" trong `sorted(arr)`. (3) `bisect_left(s, x) + 1`.

**Bài 5** — (1) Đảo chiều sắp xếp. (2) `sorted(..., reverse=True)`. (3) Phần còn lại giống bài 3.

**Bài 6** — (1) Hạng dense = số giá trị khác nhau nhỏ hơn + 1. (2) Vậy đáp án = hạng − 1. (3) `[r - 1 for r in exercise_rank(arr)]`.

## 17. Đáp án tham khảo

**Bài 1:** `[3, 1, 3, 2, 4, 1]`.

**Bài 2:** `[1, 2, 3]`, `2`, `[(1, 'a'), (2, 'b')]`.

In [24]:
def answer_rank(arr):
    rank = {v: i for i, v in enumerate(sorted(set(arr)), 1)}
    return [rank[x] for x in arr]

def answer_competition_rank(arr):
    s = sorted(arr)
    return [bisect_left(s, x) + 1 for x in arr]

def answer_rank_desc(arr):
    rank = {v: i for i, v in enumerate(sorted(set(arr), reverse=True), 1)}
    return [rank[x] for x in arr]

def answer_count_smaller_distinct(arr):
    return [r - 1 for r in answer_rank(arr)]

assert answer_rank([5, -2, 5, 0, 7, -2]) == [3, 1, 3, 2, 4, 1]
assert sorted({3, 1, 2}) == [1, 2, 3] and bisect_left([1, 3, 5], 4) == 2 and list(enumerate("ab", 1)) == [(1, 'a'), (2, 'b')]
assert answer_rank([40, 10, 20, 30]) == [4, 1, 2, 3]
assert answer_competition_rank([10, 20, 20, 30]) == [1, 2, 2, 4]
assert answer_rank_desc([10, 20, 20, 30]) == [3, 2, 2, 1]
assert answer_count_smaller_distinct([10, 20, 20, 30]) == [0, 1, 1, 2]
for _ in range(200):
    arr = [rng.randint(-4, 4) for _ in range(rng.randint(0, 15))]
    assert answer_rank(arr) == rank_brute(arr)
    assert answer_competition_rank(arr) == [sum(1 for v in arr if v < x) + 1 for x in arr]
print("Tất cả đáp án PASS!")

Tất cả đáp án PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def arrayRankTransform(self, arr: list[int]) -> list[int]:
        rank = {v: i for i, v in enumerate(sorted(set(arr)), 1)}   # nén toạ độ
        return [rank[x] for x in arr]                              # giữ thứ tự gốc
```

**Checklist:**
- [ ] **Bỏ trùng** (`set`) rồi mới đánh số.
- [ ] **Sắp xếp** sau khi `set`.
- [ ] Hạng bắt đầu từ **1** (`enumerate(..., 1)`).
- [ ] Không sửa `arr` gốc (dùng `sorted`, không `arr.sort()`).
- [ ] Đã test mảng **rỗng**, toàn trùng, số âm.
- [ ] Độ phức tạp `O(n log n)`, không dùng `O(n²)`.